In [6]:
# ================== CONFIG ==================
# Заполни под свой проект и датасет BigQuery
# ================== CONFIG ==================
BQ_JOB_PROJECT  = "fabled-zone-438910-n8"   # ← ТВОЙ проект (здесь запускаем джобы)
BQ_DATA_PROJECT = "whaleteam-495709"         # ← проект с данными Sourcify (читаем)
BQ_DATASET = "sourcify_dataset"

from google.cloud import bigquery
from google.cloud.bigquery import QueryJobConfig, ScalarQueryParameter
import requests
import re
from datetime import datetime, timezone
from typing import Optional, List, Dict, Any

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/farukh/Library/Mobile Documents/com~apple~CloudDocs/ETH/Sourcify/keys/sourcify-bq.json"


# ================== CONSTANTS ==================

SOURCIFY_API = "https://sourcify.dev/server"

WEIGHTS = {
    "has_any_verified":    0.15,
    "verification_quality":0.25,
    "documentation":       0.20,
    "activity_history":    0.15,
    "complexity":          0.15,
    "security":            0.10,
}

CHAIN_WEIGHT = {
    1:  1.0, 8453: 0.9, 10: 0.9, 42161: 0.9,
    137: 0.8, 56: 0.7, 100: 0.7, 43114: 0.7,
    250: 0.6, 11155111: 0.1, 5: 0.05, 80001: 0.05,
}
PRODUCTION_CHAINS = {1, 8453, 10, 42161, 137, 56, 100, 43114, 250}

# Опасные паттерны в исходниках
DANGER_PATTERNS = [
    r"\bselfdestruct\b",
    r"\btx\.origin\b",
    r"\bblacklist\b",
    r"function\s+rug\b",
    r"emergencyWithdraw\s*\(",
    r"onlyOwner[^}]{0,200}withdraw",
]


# ================== BIGQUERY: контракты по кошельку ==================

def get_contracts_for_deployer_bq(
    deployer: str,
    project: str = BQ_DATA_PROJECT,   # ← было BQ_PROJECT
    dataset: str = BQ_DATASET,
    max_contracts: int = 50,
) -> List[Dict[str, Any]]:
    
    # ✅ ИСПРАВЛЕНИЕ: джоб запускаем в СВОЁМ проекте
    client = bigquery.Client(project=BQ_JOB_PROJECT)

    sql = f"""
    SELECT
        LOWER(CONCAT('0x', TO_HEX(cd.address))) AS contract_address,
        cd.chain_id
    FROM `{BQ_DATA_PROJECT}.{BQ_DATASET}.public_contract_deployments` cd
    JOIN `{BQ_DATA_PROJECT}.{BQ_DATASET}.public_verified_contracts` vc
        ON vc.deployment_id = cd.id
    WHERE
        cd.deployer = FROM_HEX(REGEXP_REPLACE(LOWER(@deployer), r'^0x', ''))
    ORDER BY cd.created_at DESC
    LIMIT @limit
    """

    job_config = QueryJobConfig(
        query_parameters=[
            ScalarQueryParameter("deployer", "STRING", deployer),
            ScalarQueryParameter("limit", "INT64", max_contracts),
        ]
    )

    query_job = client.query(
        sql,
        job_config=job_config,
        location="europe-west1",  # как в UI
    )
    rows = list(query_job)

    contracts = [
        {"address": row["contract_address"], "chain_id": row["chain_id"]}
        for row in rows
    ]
    return contracts

# ================== SOURCIFY FETCH ==================

def fetch_sourcify(address: str, chain_id: int = 1) -> Optional[dict]:
    """
    Загружает полный JSON из Sourcify API v2 (?fields=all) для конкретного контракта.
    """
    url = f"{SOURCIFY_API}/v2/contract/{chain_id}/{address}"
    params = {"fields": "all"}
    try:
        r = requests.get(url, params=params, timeout=12)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

# ================== FEATURE EXTRACTION ==================

def extract_features(data: dict, chain_id: int) -> dict:
    chain_w = CHAIN_WEIGHT.get(chain_id, 0.5)
    is_prod = chain_id in PRODUCTION_CHAINS

    # Match quality
    creation_match = data.get("creationMatch") == "match"
    runtime_match  = data.get("runtimeMatch")  == "match"
    runtime_meta   = bool((data.get("runtimeBytecode") or {}).get("metadataMatch"))
    creation_meta  = bool((data.get("creationBytecode") or {}).get("metadataMatch"))
    match_score    = (creation_match * 0.3 +
                      runtime_match  * 0.5 +
                      runtime_meta   * 0.1 +
                      creation_meta  * 0.1)

    # Documentation
    devdoc      = data.get("devdoc") or {}
    userdoc     = data.get("userdoc") or {}
    storage     = data.get("storageLayout") or {}
    has_devdoc  = len(devdoc.get("methods", {})) > 0
    has_userdoc = len(userdoc.get("methods", {})) > 0 or bool(userdoc.get("notice"))
    has_storage = bool(storage.get("storage"))
    doc_score   = has_devdoc * 0.4 + has_userdoc * 0.3 + has_storage * 0.3

    # ABI complexity
    abi      = data.get("abi") or []
    n_func   = len([x for x in abi if x.get("type") == "function"])
    n_events = len([x for x in abi if x.get("type") == "event"])
    n_errors = len([x for x in abi if x.get("type") == "error"])
    complexity = min(1.0, n_func * 0.05 + n_events * 0.08 + n_errors * 0.02)

    # Security (по исходникам)
    sources  = data.get("sources") or {}
    all_code = " ".join(str(v) for v in sources.values())
    danger   = [p for p in DANGER_PATTERNS if re.search(p, all_code, re.IGNORECASE)]
    security = max(0.0, 1.0 - len(danger) * 0.35)

    # Date
    verified_at = None
    raw_date = data.get("verifiedAt")
    if raw_date:
        try:
            verified_at = datetime.fromisoformat(raw_date.replace("Z", "+00:00"))
        except Exception:
            pass

    # Meta
    name  = (data.get("compilation") or {}).get("name", "Unknown")
    proxy = (data.get("proxyResolution") or {}).get("isProxy", False)

    return {
        "chain_id": chain_id,
        "chain_weight": chain_w,
        "is_production": is_prod,
        "match_score": match_score,
        "doc_score": doc_score,
        "complexity": complexity,
        "security_score": security,
        "danger_patterns": danger,
        "has_devdoc": has_devdoc,
        "has_userdoc": has_userdoc,
        "has_storage": has_storage,
        "n_func": n_func,
        "n_events": n_events,
        "n_errors": n_errors,
        "verified_at": verified_at,
        "is_proxy": proxy,
        "name": name,
    }

# ================== AGGREGATION ==================

def aggregate(features_list: List[dict]) -> dict:
    if not features_list:
        return {}

    total = len(features_list)
    prod  = [f for f in features_list if f["is_production"]]

    def wavg(key: str) -> float:
        tw = sum(f["chain_weight"] for f in features_list)
        return sum(f[key] * f["chain_weight"] for f in features_list) / tw if tw else 0.0

    dates = [f["verified_at"] for f in features_list if f["verified_at"]]
    span_years = months_since = 0.0
    activity_score = 0.0
    if dates:
        now = datetime.now(timezone.utc)
        span_years   = (max(dates) - min(dates)).days / 365.0
        months_since = (now - max(dates)).days / 30.0
        span_score   = min(1.0, span_years / 2.0)
        recency_score = 1.0 if months_since < 12 else max(0.0, 1.0 - (months_since - 12) / 24)
        activity_score = span_score * 0.6 + recency_score * 0.4

    unique_prod_chains = sorted({f["chain_id"] for f in prod})
    multichain_bonus   = min(0.3, (len(unique_prod_chains) - 1) * 0.15) if prod else 0.0
    proxy_bonus        = 0.1 if any(f["is_proxy"] for f in features_list) else 0.0

    return {
        "total": total,
        "n_prod": len(prod),
        "unique_prod_chains": unique_prod_chains,
        "avg_match": wavg("match_score"),
        "avg_doc":   wavg("doc_score"),
        "avg_cx":    wavg("complexity"),
        "avg_sec":   wavg("security_score"),
        "activity_score": activity_score,
        "span_years": span_years,
        "months_since_last": months_since,
        "multichain_bonus": multichain_bonus,
        "proxy_bonus": proxy_bonus,
        "all_danger": list({p for f in features_list for p in f["danger_patterns"]}),
    }

# ================== SCORING ==================

def compute_score(agg: dict):
    if not agg or agg["total"] == 0:
        empty = {k: {"score": 0.0, "max": w, "note": "No data"} for k, w in WEIGHTS.items()}
        return 0.0, empty

    bk = {}

    # 1. has_any_verified
    count_factor = min(1.0, agg["total"] / 5.0)
    s1 = (0.6 + count_factor * 0.4) * WEIGHTS["has_any_verified"]
    bk["has_any_verified"] = {
        "score": round(s1, 4),
        "max": WEIGHTS["has_any_verified"],
        "note": f"{agg['total']} contracts ({agg['n_prod']} production)",
    }

    # 2. verification_quality
    s2 = agg["avg_match"] * WEIGHTS["verification_quality"]
    bk["verification_quality"] = {
        "score": round(s2, 4),
        "max": WEIGHTS["verification_quality"],
        "note": f"Weighted match quality: {agg['avg_match']:.2f}",
    }

    # 3. documentation
    s3 = agg["avg_doc"] * WEIGHTS["documentation"]
    bk["documentation"] = {
        "score": round(s3, 4),
        "max": WEIGHTS["documentation"],
        "note": f"Doc quality: {agg['avg_doc']:.2f} (devdoc/userdoc/storageLayout)",
    }

    # 4. activity_history
    s4 = agg["activity_score"] * WEIGHTS["activity_history"]
    bk["activity_history"] = {
        "score": round(s4, 4),
        "max": WEIGHTS["activity_history"],
        "note": f"{agg['span_years']:.1f}yr span, last {agg['months_since_last']:.0f}mo ago",
    }

    # 5. complexity (+ multichain)
    s5 = min(
        WEIGHTS["complexity"],
        (agg["avg_cx"] + agg["multichain_bonus"]) * WEIGHTS["complexity"],
    )
    bk["complexity"] = {
        "score": round(s5, 4),
        "max": WEIGHTS["complexity"],
        "note": f"Complexity: {agg['avg_cx']:.2f}, prod chains: {agg['unique_prod_chains']}",
    }

    # 6. security
    s6 = agg["avg_sec"] * WEIGHTS["security"]
    danger_note = (
        f"Danger found: {agg['all_danger']}" if agg["all_danger"] else "Clean — no dangerous patterns"
    )
    bk["security"] = {
        "score": round(s6, 4),
        "max": WEIGHTS["security"],
        "note": danger_note,
    }

    total = min(1.0, sum(v["score"] for v in bk.values()) + agg["proxy_bonus"] * 0.02)
    return round(total, 3), bk

In [5]:
from google.cloud import bigquery

client = bigquery.Client(project=BQ_JOB_PROJECT)

sql = """
SELECT
  chain_id,
  COUNT(*) AS n
FROM `whaleteam-495709.sourcify_dataset.public_contract_deployments`
GROUP BY chain_id
LIMIT 5
"""

for row in client.query(sql, location="europe-west1"):
    print(row)

Row((421614, 89421), {'chain_id': 0, 'n': 1})
Row((11155420, 110899), {'chain_id': 0, 'n': 1})
Row((146, 582), {'chain_id': 0, 'n': 1})
Row((40, 1854), {'chain_id': 0, 'n': 1})
Row((167000, 94), {'chain_id': 0, 'n': 1})


In [7]:
# ================== PRETTY PRINT ==================

def print_result(result: dict):
    score   = result["score"]
    verdict = result["verdict"]
    badge   = {"APPROVE": "✅", "REVIEW": "⚠️", "REJECT": "❌"}[verdict]

    print(f"\n{'═'*62}")
    print(f"  SOURCIFY RELIABILITY AUDIT")
    print(f"  Developer: {result['wallet']}")
    print(f"{'═'*62}")
    print(f"  Score:   {score:.3f} / 1.000   {badge} {verdict}")
    print(f"\n  Breakdown by criterion:")
    print(f"  {'Criterion':<26} {'Fill':22} {'Got':>5} / {'Max':>4}")
    print(f"  {'─'*26} {'─'*22} {'─'*5}   {'─'*4}")
    for k, v in result["breakdown"].items():
        fill = int(v["score"] / v["max"] * 20) if v["max"] else 0
        bar  = "█" * fill + "░" * (20 - fill)
        print(f"  {k:<26} [{bar}] {v['score']:>5.3f} / {v['max']:.2f}")
        print(f"  {'':26}   ↳ {v['note']}")
    print(f"\n  Summary:")
    for s in result["summary"]:
        print(f"    • {s}")
    print(f"{'═'*62}\n")

# ================== MAIN FUNCTION ==================

def audit_developer(
    deployer_address: str,
    max_contracts: int = 50,
    project: str = BQ_DATA_PROJECT,   # ← было BQ_PROJECT
    dataset: str = BQ_DATASET,
    verbose: bool = True,
) -> dict:
    """
    Главная функция для твоей части.
    1) Берёт адрес кошелька-разработчика
    2) Через BigQuery достаёт все его verified-контракты
    3) Для каждого вызывает Sourcify API
    4) Считает скор 0–1 и даёт разбор по критериям
    """
    if verbose:
        print(f"\n=== AUDIT {deployer_address} ===")
        print("[1/4] Fetching contracts from BigQuery...")

    contracts = get_contracts_for_deployer_bq(
        deployer=deployer_address,
        project=project,
        dataset=dataset,
        max_contracts=max_contracts,
    )

    if verbose:
        print(f"      Found {len(contracts)} verified contract(s) in BigQuery")

    features_list = []
    for c in contracts:
        data = fetch_sourcify(c["address"], c["chain_id"])
        if data:
            f = extract_features(data, c["chain_id"])
            features_list.append(f)
            if verbose:
                print(f"  ✅ {c['address'][:12]}... — {f['name']} (chain {c['chain_id']})")
        else:
            if verbose:
                print(f"  ❌ {c['address'][:12]}... — not in Sourcify")

    if verbose:
        print("\n[3/4] Aggregating features & computing score...")

    agg = aggregate(features_list)
    score, breakdown = compute_score(agg)

    verdict = "APPROVE" if score >= 0.65 else ("REVIEW" if score >= 0.35 else "REJECT")

    summary = []
    if agg:
        summary.append(f"{agg['total']} verified contracts ({agg['n_prod']} on mainnet/L2)")
        if agg["span_years"] > 0:
            summary.append(f"Activity history: {agg['span_years']:.1f} years")
        if agg["months_since_last"] < 999:
            summary.append(f"Last verified ~{agg['months_since_last']:.0f} months ago")
        summary.append("Good documentation" if agg["avg_doc"] > 0.5 else "Weak documentation")
        summary.append(
            "Clean code ✓" if not agg["all_danger"] else f"⚠️ Dangerous patterns: {agg['all_danger']}"
        )
        if len(agg["unique_prod_chains"]) > 1:
            summary.append(f"Multi-chain developer: chains {agg['unique_prod_chains']}")
    else:
        summary.append("No verified contracts found in Sourcify — REJECT")

    result = {
        "wallet": deployer_address,
        "score": score,
        "verdict": verdict,
        "breakdown": breakdown,
        "summary": summary,
    }

    if verbose:
        print_result(result)

    return result



res = audit_developer(
    "0x41653c7d61609D856f29355E404F310Ec4142Cfb",  # Uniswap deployer
    max_contracts=30,
    verbose=True
)


=== AUDIT 0x41653c7d61609D856f29355E404F310Ec4142Cfb ===
[1/4] Fetching contracts from BigQuery...
      Found 30 verified contract(s) in BigQuery
  ✅ 0x1f9840a85d... — UNI (chain 7777777)
  ✅ 0x1a9c8182c0... — Recover (chain 42220)
  ✅ 0x1f9840a85d... — Recover (chain 42220)
  ✅ 0x1a9c8182c0... — Recover (chain 42170)
  ✅ 0x1f9840a85d... — Recover (chain 42170)
  ✅ 0x1f9840a85d... — UNI (chain 1868)
  ✅ 0x4b4e140d1f... — TreasuryVester (chain 369)
  ✅ 0xe3953d9d31... — TreasuryVester (chain 369)
  ✅ 0x4b4e140d1f... — TreasuryVester (chain 1)
  ✅ 0xe3953d9d31... — TreasuryVester (chain 1)
  ✅ 0xa1484c3aa2... — StakingRewards (chain 369)
  ✅ 0xca35e32e79... — StakingRewards (chain 369)
  ✅ 0x7fba4b8dc5... — StakingRewards (chain 369)
  ✅ 0x6c3e4cb2e9... — StakingRewards (chain 369)
  ✅ 0x18e433c7bf... — FeeToSetter (chain 369)
  ✅ 0xdaf819c243... — FeeTo (chain 369)
  ✅ 0x3d30b1ab88... — TreasuryVester (chain 369)
  ✅ 0x4750c43867... — TreasuryVester (chain 369)
  ✅ 0x3032ab3fa8... — S

## Формула итогового score

### Веса критериев

| Критерий | Вес |
|---|---|
| `has_any_verified` | 0.15 |
| `verification_quality` | 0.25 |
| `documentation` | 0.20 |
| `activity_history` | 0.15 |
| `complexity` | 0.15 |
| `security` | 0.10 |

---

### 1. Наличие верифицированных контрактов

$$\text{count\_factor} = \min\!\left(1,\ \frac{\texttt{agg["total"]}}{5}\right)$$

$$S_{\text{hav}} = \bigl(0.6 + 0.4 \times \text{count\_factor}\bigr) \times 0.15$$

---

### 2. Качество верификации

$$S_{\text{ver}} = \texttt{agg["avg\_match"]} \times 0.25$$

---

### 3. Документация (devdoc / userdoc / storageLayout)

$$S_{\text{doc}} = \texttt{agg["avg\_doc"]} \times 0.20$$

---

### 4. История и свежесть активности

$$S_{\text{act}} = \texttt{agg["activity\_score"]} \times 0.15$$

---

### 5. Сложность + мультичейн

$$S_{\text{cx,raw}} = \bigl(\texttt{agg["avg\_cx"]} + \texttt{agg["multichain\_bonus"]}\bigr) \times 0.15$$

$$S_{\text{cx}} = \min\!\left(0.15,\ S_{\text{cx,raw}}\right)$$

---

### 6. Безопасность кода

$$S_{\text{sec}} = \texttt{agg["avg\_sec"]} \times 0.10$$

---

### 7. Итоговый скор

Бонус за прокси:

$$S_{\text{proxy}} = \texttt{agg["proxy\_bonus"]} \times 0.02$$

Финальное значение от 0 до 1:

$$\boxed{\text{score} = \min\!\left(1,\ S_{\text{hav}} + S_{\text{ver}} + S_{\text{doc}} + S_{\text{act}} + S_{\text{cx}} + S_{\text{sec}} + S_{\text{proxy}}\right)}$$

---
---
---

Почему именно эти колонки и такие веса
1. has_any_verified (0.15)
Сигналы: количество verified‑контрактов agg["total"], количество прод‑контрактов agg["n_prod"].

Логика: если у кошелька вообще нет verified‑контрактов — это почти всегда новичок/скимер, его сразу надо сильно штрафовать. Уже 3–5 контрактов резко повышают доверие, дальше отдаём «слово» другим критериям.

Вес 0.15: это «входной билет» в систему — важный, но не решающий, чтобы не поощрять фарминг пустых контрактов.

2. verification_quality (0.25)
Сигналы: creationMatch, runtimeMatch, metadataMatch → агрегируется в match_score, потом среднее agg["avg_match"].

Логика: полнота и качество верификации — сильный прокси того, насколько аккуратно разработчик обращается с кодом и ставит ли он целью прозрачность.

Вес 0.25: один из двух самых сильных критериев (вместе с документацией) — мы поощряем тех, кто не просто деплоит, но и нормально верифицирует контракты.

3. documentation (0.20)
Сигналы: has_devdoc, has_userdoc, has_storage → складываются в doc_score, далее agg["avg_doc"].

Логика: devdoc/userdoc + storageLayout — это признак инженерной культуры, удобного аудита и апгрейда. Без документации проект сложнее поддерживать и проверять.

Вес 0.20: чуть меньше, чем match‑quality, потому что для некоторых простых контрактов документация менее критична, но всё равно важна.

4. activity_history (0.15)
Сигналы: span_years (разброс между первым и последним verified‑контрактом), months_since_last (давность последнего), агрегируется в activity_score.

Логика: нам важен не только факт, что человек что‑то деплоил, но и что он делает это не разово, а в течение времени и недавно.

Вес 0.15: на уровне «наличия контрактов» — усиливает доверие к старым и активным разработчикам, но не убивает новичков, если по остальным метрикам всё ок.

5. complexity (0.15)
Сигналы: количество функций/ивентов/ошибок n_func, n_events, n_errors → complexity, плюс unique_prod_chains → multichain_bonus.

Логика: сложные контракты и мультичейн‑деплой свидетельствуют о более серьёзной инженерной нагрузке и доверии к разработчику. Но слишком высокий вес может поощрять «раздутые» контракты.

Вес 0.15 + кап по формуле: это важный плюс, но с ограничением, чтобы сложность не перестала быть преимуществом и не стала «эксплойтом» для набивания скора.

6. security (0.10)
Сигналы: danger_patterns (наличие selfdestruct, tx.origin, emergencyWithdraw, onlyOwner withdraw и т.д.), агрегируется в security_score.

Логика: наличие опасных паттернов — красный флаг, отсутствие — базовый гигиенический плюс.

Вес 0.10: это «минусовой» критерий — главное, чтобы не было плохого; поэтому поощрение умеренное, а штраф — сильный (через формулу 1 - 0.35 * len(danger)).

7. Бонусы multichain_bonus и proxy_bonus
multichain_bonus до 0.3: небольшое усиление для тех, кто реально деплоит на нескольких прод‑сетях. Не отдельный критерий, а добавка к сложности, чтобы не плодить ещё одну колонку.

proxy_bonus 0.1 → переводится в максимум 0.002 к общему скору: показывает, что кто‑то умеет работать с прокси/апгрейдами, но это всё же тонкий сигнал, не тянущий на отдельный критерий.

---
---
---

Какие сигналы мы пока не используем и почему

Из Sourcify BigQuery (структурно доступны, но не в скоре напрямую)

compiler, version, language, fullyQualifiedName из publiccompiledcontracts.

Причина: сильно связаны с типом проекта и стандартами экосистемы (например, почти все используют solc 0.8.x и OpenZeppelin) — больше шум, чем сигнал для общих грантов. Можно добавить позже как эвристику для слишком старых версий.



Детали compilerSettings (evmVersion, optimizer.runs, viaIR, metadata.bytecodeHash).

Причина: полезно для deep‑аудита, но сложно объяснить простому пользователю и мало влияет на доверие команды к разработчику на уровне «дать грант / не дать».



blockNumber, transactionHash, transactionIndex из public_contract_deployments.

Причина: мы уже используем время через createdAt/verifiedAt и span/recency; конкретные номера блоков не дают дополнительного смыслового сигнала для человека.



source paths и полный текст из publiccompiledcontractssources и publicsources (путь к файлам, полный Solidity‑код).

Причина: это очень тяжёлые данные; мы вместо этого берём только бинарный сигнал «есть ли devdoc/userdoc/storageLayout» и несколько опасных паттернов. Всё остальное можно докинуь потом через отдельный линтер/LLM‑аудит исходников, но для MVP это слишком жирно по времени и стоимости.



Таблицы publiccontracts, publiccode, publicsignatures, publiccompiledcontractssignatures, publicsourcifymatches.

Причина: они нужны для продвинутых вещей типа анализа байткода, функций‑админок, подписи событий. В MVP мы берём только то, что легко объясняется пользователю (верификация, дока, активность, базовый security‑паттерн), чтобы скор выглядел понятным, а не магическим числом.



Внутренние фичи, которые используются только в текстовом объяснении, но не в весах
unique_prod_chains — список прод‑цепочек; напрямую не идёт в формулу, используется для бонуса и для Summary.



span_years и months_since_last — используются только для вычисления activity_score и потом появляются в текстовых нотах.



danger_patterns (список регекспов) — участвует в security_score, но детали паттернов идут только в текст, а не в отдельные веса по каждому паттерну.

---
---
---

# All Big Query columns:

In [10]:
from google.cloud import bigquery

client = bigquery.Client(project=BQ_JOB_PROJECT)

schema_sql = """
SELECT table_name, column_name
FROM `whaleteam-495709.sourcify_dataset.INFORMATION_SCHEMA.COLUMNS`
ORDER BY table_name, ordinal_position
"""

for row in client.query(schema_sql, location="europe-west1"):
    print(row.table_name, ":", row.column_name)

public_code : code_hash
public_code : code
public_code : code_hash_keccak
public_code : created_at
public_code : updated_at
public_code : created_by
public_code : updated_by
public_code : datastream_metadata
public_compiled_contracts : id
public_compiled_contracts : created_at
public_compiled_contracts : updated_at
public_compiled_contracts : created_by
public_compiled_contracts : updated_by
public_compiled_contracts : compiler
public_compiled_contracts : version
public_compiled_contracts : language
public_compiled_contracts : name
public_compiled_contracts : fully_qualified_name
public_compiled_contracts : compiler_settings
public_compiled_contracts : compilation_artifacts
public_compiled_contracts : creation_code_hash
public_compiled_contracts : creation_code_artifacts
public_compiled_contracts : runtime_code_hash
public_compiled_contracts : runtime_code_artifacts
public_compiled_contracts : datastream_metadata
public_compiled_contracts : additional_input
public_compiled_contracts_si